In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient
import pprint
import requests
from bs4 import BeautifulSoup

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

# httpリクエスト時のUA（ブラウザでリクエスト時の値を記録し設定）
user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'

In [2]:
# ツール定義
def define_tools():
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "ネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        }),
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_txt_from_url",
                "description": "指定したURLの、.TXTの内容を文字列で取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {"type": "string", "description": "URL"},
                    },
                    "required": ["url"],
                },
            },
        }),
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_html_body_from_url",
                "description": "指定したURLのHTMLから、BODY部分を文字列で取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {"type": "string", "description": "URL"},
                    },
                    "required": ["url"],
                },
            },
        })
    ]

In [3]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]}, indent=4, ensure_ascii=False)

In [4]:
# URLから.txtを取得し文字列として返す関数の作成
def get_txt_from_url(url):
    # ページのHTMLを取得
    response = requests.get(url, headers={"User-Agent": user_agent})
    response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定
 
    return response.text

In [5]:
# URLからHTMLを取得しBODY部分を文字列として返す関数の作成
def get_html_body_from_url(url):
    # ページのHTMLを取得
    response = requests.get(url, headers={"User-Agent": user_agent})
    response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定
 
    # HTMLをBeautifulSoupでパースし、body部分を取り出す
    soup = BeautifulSoup(response.text, "html.parser")
    body_html = str(soup.body)  # body部分のHTMLを文字列として取得
    
    return body_html

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, messages, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)
    
    with open("log.txt", mode="a", encoding="utf-8") as file:
        file.write("\nfunction_response=")
        pprint.pprint(function_response, stream = file)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages + [
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# 言語モデルへの質問を行う関数
def ask_question(messages, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    return response

In [8]:
# ユーザーからの質問を処理する関数
def process_response(messages, tools, question):
    response = ask_question(messages, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        with open("log.txt", mode="a", encoding="utf-8") as file:
            file.write("\n\nツール呼出発生\n\nresponse=")
            pprint.pprint(vars(response), stream = file)
        final_response = handle_tool_call(response, messages, question)
        # ログに追記
        with open("log.txt", mode="a", encoding="utf-8") as file:
            file.write("\nfinal_response=")
            pprint.pprint(vars(final_response), stream = file)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [13]:
def main():
    with open("log.txt", mode="w", encoding="utf-8") as file: None

    # チャットボットへの組み込み
    tools = define_tools()

    # メッセージを格納するリスト
    messages=[]
    
    while(True):
        # ユーザーからの質問を受付
        question = input("メッセージを入力:")
        # 質問が入力されなければ終了
        if question.strip()=="":
            break
        print(f"\n質問:{question}")

        # メッセージにユーザーからの質問を追加
        messages.append({"role": "user", "content": question.strip()})
        # やりとりが8を超えたら古いメッセージから削除
        # やりとりが8個以下になっても 先頭のメッセージがユーザの質問になるまでは 削除を続行
        while len(messages) > 8 or messages[0]["role"] != "user":
            del_message = messages.pop(0)

        # ログに追記
        with open("log.txt", mode="a", encoding="utf-8") as file:
            file.write(f"ユーザー入力：{question}")
        
        # 言語モデルに質問
        response_message = process_response(messages, tools, question)

        # メッセージに言語モデルからの回答を追加
        print(response_message, flush=True)
        messages.append({"role": "assistant", "content": response_message})

        # ログに追記
        with open("log.txt", mode="a", encoding="utf-8") as file:
            file.write("\n\nmessages=")
            json.dump(messages, file, indent=4, ensure_ascii=False)
            file.write("\n\n" + "#"*50 + "\n\n")

    print("\n---ご利用ありがとうございました！---")

In [15]:
main()


質問:「https://www.microsoft.com/robots.txt」の内容から、「https://www.microsoft.com/ja-jp/servicesagreement」のスクレイピングが許可されているか判断して
「https://www.microsoft.com/robots.txt」の内容を分析したところ、特定のURLに対するスクレイピングの許可/不許可については、以下のポイントが考慮されます。

1. **User-agent: ***: すべてのクローラーに適用されるルールです。
2. **Disallow**: 特定のパスに対してアクセスを禁止していますが、これには「https://www.microsoft.com/ja-jp/servicesagreement」は含まれていません。

「https://www.microsoft.com/ja-jp/servicesagreement」に関する具体的な禁止規定は記載されていないので、このURLへのスクレイピングは許可されていると判断できます。

質問:「https://www.microsoft.com/ja-jp/servicesagreement」の内容から、「https://visualstudio.microsoft.com/ja/」のスクレイピングが許可されているか判断して
私は直接的にウェブサイトをスクレイピングすることができませんが、「https://www.microsoft.com/ja-jp/servicesagreement」のサービス契約ページから、特定のサービスに対して利用者が従うべきルールや条件を確認することができます。

このページに記載されている内容を読む限り、スクレイピングに関して特に明言されている制限がないか、または一般に禁止されているサービスのリストに「https://visualstudio.microsoft.com/ja/」が含まれていない限り、スクレイピングが許可されている可能性があります。

ただし、Microsoftのサービス利用規約は非常に詳細であり、特定の条件や例外が存在するため、具体的なスクレイピングを行う前には、必ず規約を再度確認してください。

総括すると、具体的な禁止がなければ、スクレイピングが許可されていると考

In [16]:
main()


質問:「https://www.microsoft.com/ja-jp/servicesagreement」に記載されている規約から、「https://visualstudio.microsoft.com/ja/」のスクレイピングは禁止されているか判断して
「https://www.microsoft.com/ja-jp/servicesagreement」のサービス規約には、以下のような内容が含まれています。

1. **倫理規定** のセクションにおいて、「他のユーザーによるこれらの規則違反に力を貸してはならない」と明記されています。また、"スクレイピング"（データの自動収集行為）に対しては、AI システムや「不正な方法」でのアクセスを試みないことが求められています。特に、「本サービスへのアクセス、使用、その利用可否に関する制限を回避してはなりません (AI システムや容認できないスクレイピングの「ジェイルブレイク」を試みるなど)」という具体的な禁止事項があります。

したがって、Visual Studioのウェブサイト「https://visualstudio.microsoft.com/ja/」のスクレイピングは、禁止されていると考えられます。規約に従う限り、Scraping 行為はオフリミットです。企業としての遵守が求められるため、スクレイピングを計画している場合は、これを回避する必要があります。

---ご利用ありがとうございました！---
